In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime

In [ ]:
#Data collection


In [ ]:
now= datetime.now()
start=datetime(now.year-10,now.month,now.day)
end= now
ticker='AAPL'
df=yf.download(ticker,start,end)
df

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
#2> Data exploration and visulisation

In [ ]:
df

In [ ]:
type(df)

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.head()

In [ ]:
df=df.reset_index()

In [ ]:
df.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.Close)
plt.title(ticker)
plt.xlabel('Days')
plt.ylabel('Close price')

In [ ]:
#3> Feature Engineering


In [ ]:
# temp_df=[10,20,30,40,50,60,70,80,90,100]
# print(sum(temp_df[2:7])/5)

In [ ]:
df1= pd.DataFrame([10,20,30,40,50,60,70,80,90,100])
df1

In [ ]:
df1['MA_5']= df1.rolling(5).mean()
df1

In [ ]:
#100 days moving average

In [ ]:
df['MA_100']=df.Close.rolling(100).mean()
df.head()

In [ ]:
df.head(103)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.Close)
plt.plot(df['MA_100'],'r')
plt.title('100 days moving average')
plt.xlabel('Days')
plt.ylabel('Close price')


In [ ]:
#Moving averages of 200

In [ ]:
df['MA_200']=df.Close.rolling(200).mean()
df.head(203)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.Close)
plt.plot(df['MA_100'],'r')
plt.plot(df['MA_200'],'g')
plt.title('200 days moving average')
plt.xlabel('Days')
plt.ylabel('price')

In [ ]:
df

In [ ]:
#calculating % changed in each trading session

In [ ]:
df['percentage Changed']= df.Close.pct_change()
df[['Close','percentage Changed']]

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df['percentage Changed'])

In [ ]:
# 4> Data Preprocessing

In [ ]:
df.shape

In [ ]:
#splitting data into training and testing
data_training=pd.DataFrame(df.Close[0:int(len(df)*0.7)])
data_testing=pd.DataFrame(df.Close[int(len(df)*0.7):int(len(df))])
print(data_training)
print(data_testing)

In [ ]:
data_training

In [ ]:
data_testing

In [ ]:
# Scaling down the data between 0 and 1

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler= MinMaxScaler(feature_range=(0,1))

In [ ]:
data_training_array=scaler.fit_transform(data_training)
print(data_training_array)

In [ ]:
type(data_training_array)

In [ ]:
#5> sequence creation
#lstm

In [ ]:
x_train=[]
y_train=[]

for i in range (100,data_training_array.shape[0]):
    x_train.append(data_training_array[i-100:i])
    y_train.append(data_training_array[i,0])

x_train,y_train= np.array(x_train),np.array(y_train)

In [ ]:
x_train

In [ ]:
#6> Model Building
import tensorflow as tf
print(tf.__version__)

In [ ]:
#ml model
from keras.models import Sequential
from keras.layers import Dense, LSTM, Input

In [ ]:
model = Sequential()

model.add(Input(shape=(100,1)))
model.add(LSTM(units=128,activation='tanh',return_sequences=True))
model.add(LSTM(units=64))
model.add(Dense(25))
model.add(Dense(1))

In [ ]:
#7 model Training

In [ ]:
model.compile(optimizer='adam',loss='mean_squared_error')
model.fit(x_train,y_train,epochs=50)

In [ ]:
model.summary()

In [ ]:
model.save('stock_prediction_model.keras')

In [ ]:
#8>preparing test data

In [ ]:
data_training

In [ ]:
data_testing

In [ ]:
past_100_days= data_training.tail(100)

In [ ]:
past_100_days

In [ ]:
final_df=pd.concat([past_100_days,data_testing],ignore_index=True)
final_df

In [ ]:
input_data=scaler.fit_transform(final_df)
input_data

In [ ]:
input_data.shape

In [ ]:
#LSTM MODELS EXPECTS SEQUENTIAL DATA
x_test=[]
y_test=[]

for i in range(100,input_data.shape[0]):
   x_test.append(input_data[i-100:i])
   y_test.append(input_data[i,0])

In [ ]:
x_test,y_test=np.array(x_test),np.array(y_test)

In [ ]:
x_test[0].shape

In [ ]:
x_test

In [ ]:
#Making Predictions

In [ ]:
y_predicted=model.predict(x_test)

In [ ]:
y_predicted

In [ ]:
y_test

In [ ]:
y_predicted=scaler.inverse_transform(y_predicted.reshape(-1,1)).flatten()
y_test=scaler.inverse_transform(y_test.reshape(-1,1)).flatten()

In [ ]:
y_predicted

In [ ]:
y_test

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(y_test,'b',label='original price')
plt.plot(y_predicted,'r',label='Predicted Price')
plt.xlabel('Days')
plt.ylabel('Price')
plt.legend()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(y_test,'b',label='original price')
plt.plot(y_predicted,'r',label='Predicted Price')
plt.xlabel('Days')
plt.ylabel('Price')
plt.legend()
plt.xlim(450,720)
plt.ylim(140,300)

In [ ]:
#model Evaluation

In [ ]:
#Mean squared Error(mse)

In [ ]:
from sklearn.metrics import mean_squared_erro,r2_score

In [ ]:
mse=mean_squared_error(y_test,y_predicted)
print(f"Mean squared Error (MSE):{mse}")

In [ ]:
#Root MEAN Squared error
rmse=np.sqrt(mse)
print(f"Mean squared Error (MSE):{rmse}")

In [ ]:
#r-squared
r2=r2_score(y_test,y_predicted)
print(f"R-sqaured:{r2}")